# 02 - Feature engineering and chronological split

Build leakage-safe features, persist the full feature table, and create chronological train/validation/test partitions. Requires notebook 01 to have completed successfully.

In [1]:
## Setup

import os
os.environ.pop("SPARK_HOME", None)  # Use the JARs bundled with the installed PySpark version.

import sys, warnings, yaml
from pathlib import Path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import numpy as np
import pandas as pd
import plotly.graph_objects as go

_cwd = Path.cwd().resolve()
_candidates = list(dict.fromkeys([
    _cwd,
    *_cwd.parents,
    *(path.parent.parent for path in _cwd.glob("*/configs/pipeline.yaml")),
]))
_required_project_files = (
    "configs/pipeline.yaml",
    "src/features/pipeline.py",
    "notebooks/01_data_pipeline.ipynb",
    "Home.py",
)
_project_roots = [
    path for path in _candidates
    if all((path / relative).is_file() for relative in _required_project_files)
]
if not _project_roots:
    raise FileNotFoundError(
        "Could not identify this project. Open its repository root or set the notebook working directory to it."
    )
if len(_project_roots) > 1:
    raise RuntimeError(
        f"Multiple project roots found: {_project_roots}. Open the intended repository root."
    )
PROJECT_ROOT = _project_roots[0]
os.chdir(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.spark_session   import get_spark
from src.features.pipeline    import build_feature_pipeline
from src.models.splitter      import chronological_split
from src.visualisation.eda_plots import plot_correlation_heatmap

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED  = PROJECT_ROOT / CFG['artifacts']['processed_dir']
FIGURES    = PROJECT_ROOT / CFG['reports']['figures_dir']
for d in [PROCESSED, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)
with open(PROCESSED / 'audit_meta.yaml') as f:
    META = yaml.safe_load(f)

TS_COL     = META['ts_col']
TARGET_COL = f'target_log_return_{CFG["features"]["target_horizon"]}'

print(f'Timestamp column : {TS_COL}')
print(f'Target column    : {TARGET_COL}')

Timestamp column : timestamp
Target column    : target_log_return_1


---
## 1 · Feature Engineering

> ⚠️ **Anti-leakage**: all window functions use `rowsBetween(-(w-1), 0)` (past-only). Target label built with `F.lead()`.

In [2]:
# ── Load cleaned data → Spark ─────────────────────────────────────────
df_clean = pd.read_parquet(PROCESSED / 'btc_cleaned.parquet')
if TS_COL not in df_clean.columns or 'close' not in df_clean.columns:
    raise ValueError(f'Cleaned data must contain timestamp {TS_COL!r} and close columns.')
df_clean[TS_COL] = pd.to_datetime(df_clean[TS_COL])

# Keep only raw fields consumed by the feature pipeline; provider indicators may leak future data.
raw_candidates = [
    TS_COL, 'open', 'high', 'low', 'close', 'volume',
    'number_of_trades', 'taker_buy_base_asset_volume',
]
input_cols = [column for column in raw_candidates if column in df_clean.columns]
df_clean = df_clean.loc[:, input_cols]

spark = get_spark()
sdf = spark.createDataFrame(df_clean)
SOURCE_COLUMNS = set(sdf.columns)
print(f'Loaded raw inputs into Spark: {sdf.count():,} rows × {len(sdf.columns)} cols')

[spark_session] JAVA_HOME pinned → C:\Program Files\Java\jdk-17
Loaded raw inputs into Spark: 800,000 rows × 6 cols


In [3]:
# ── Run full feature pipeline ─────────────────────────────────────────
try:
    interval_seconds = float(META['expected_interval_seconds'])
except (KeyError, TypeError, ValueError) as exc:
    raise ValueError('Audit metadata must include expected_interval_seconds.') from exc
if not np.isfinite(interval_seconds) or interval_seconds <= 0:
    raise ValueError(f'Invalid audited bar interval: {interval_seconds!r}')

SECONDS_PER_YEAR = 365.25 * 24 * 60 * 60
ANN_FACTOR = float(np.sqrt(SECONDS_PER_YEAR / interval_seconds))
print(f'Building features (PySpark window functions; annualization factor {ANN_FACTOR:.2f})...')
sdf_features = build_feature_pipeline(
    sdf, ts_col=TS_COL, close_col='close', ann_factor=ANN_FACTOR,
)

print(f'Feature DataFrame: {len(sdf_features.columns)} columns')

Building features (PySpark window functions; annualization factor 725.23)...


Feature DataFrame: 69 columns


In [4]:
# ── Persist features without collecting the full dataset ───────────────
feature_cols = [c for c in sdf_features.columns
                if c not in SOURCE_COLUMNS and not c.startswith('target_')]
feature_path = PROCESSED / 'btc_features.parquet'
sdf_features.write.mode('overwrite').parquet(str(feature_path))
sdf_features = spark.read.parquet(str(feature_path))

print(f'\nFeature groups:')
for label, kw in [('Returns', 'return'), ('Volatility', 'vol'),
                  ('Volume', 'volume'), ('Market Structure', 'drawdown momentum slope range')]:
    cols = [c for c in feature_cols if any(k in c for k in kw.split())]
    print(f'  {label:<18}: {len(cols)} cols')
print(f'  Total features  : {len(feature_cols)}')


Feature groups:
  Returns           : 11 cols
  Volatility        : 35 cols
  Volume            : 11 cols
  Market Structure  : 16 cols
  Total features  : 62


In [5]:
# ── Feature × target correlation heatmap (top 20) ────────────────────
top_cols = feature_cols[:20] + [TARGET_COL]
corr_sample = (sdf_features.select(*top_cols)
               .sample(withReplacement=False, fraction=0.05, seed=42)
               .limit(50000).toPandas())
fig = plot_correlation_heatmap(corr_sample, columns=top_cols,
                               title='Feature × Target Correlation (top 20 features)')
fig.write_html(str(FIGURES / 'feature_correlation.html'), include_plotlyjs='cdn')


In [6]:
print(f'Saved Spark feature dataset → {feature_path}  '
      f'({len(sdf_features.columns)} columns)')

Saved Spark feature dataset → C:\Users\hp\Documents\GitHub\Crypto_TimeSeries_Analysis\processed\btc_features.parquet  (69 columns)


---
## 2 · Chronological Train / Val / Test Split

**Rule:** Strictly chronological — no random shuffling, ever.

In [7]:
# ── Split ─────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

TARGET_HORIZON = int(CFG['features']['target_horizon'])
train_df, val_df, test_df = chronological_split(sdf_features, ts_col=TS_COL)

def purge_partition_tail(partition, horizon):
    row_count = partition.count()
    if row_count <= horizon:
        raise ValueError(f'Partition has {row_count} rows, too few to purge {horizon} target-overlap rows.')
    cutoff = (
        partition.select(TS_COL)
        .orderBy(TS_COL)
        .limit(row_count - horizon)
        .agg(F.max(TS_COL).alias('cutoff'))
        .first()['cutoff']
    )
    purged = partition.filter(F.col(TS_COL) <= F.lit(cutoff))
    return purged, row_count - purged.count()

train_df, purged_train_rows = purge_partition_tail(train_df, TARGET_HORIZON)
val_df, purged_val_rows = purge_partition_tail(val_df, TARGET_HORIZON)
print(f'Purged target-overlap rows: train={purged_train_rows}, validation={purged_val_rows}')

print('\nSplit summary:')
for name, part in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    bounds = part.agg(F.min(TS_COL).alias('start'), F.max(TS_COL).alias('end')).first()
    print(f'  {name:5}: {bounds.start.date()} → {bounds.end.date()}  '
          f'({part.count():,} rows)')

Purged target-overlap rows: train=1, validation=1

Split summary:
  Train: 2017-08-18 → 2019-07-19  (559,998 rows)
  Val  : 2019-07-19 → 2019-12-16  (119,999 rows)
  Test : 2019-12-16 → 2020-05-14  (120,000 rows)


In [8]:
# ── Prepare Spark partitions (drop NaN rows) ──────────────────────────
all_model_cols = feature_cols + [TARGET_COL, TS_COL]
train_clean = train_df.select(*all_model_cols).dropna(subset=all_model_cols)
val_clean   = val_df.select(*all_model_cols).dropna(subset=all_model_cols)
test_clean  = test_df.select(*all_model_cols).dropna(subset=all_model_cols)

for name, part in [('Train', train_clean), ('Val', val_clean), ('Test', test_clean)]:
    print(f'{name}: {part.count():,} rows × {len(all_model_cols)} columns')

Train: 558,601 rows × 64 columns
Val: 119,999 rows × 64 columns
Test: 120,000 rows × 64 columns


In [9]:
# ── Target distribution per partition ────────────────────────────────
fig = go.Figure()
for seed, (name, part, col) in enumerate([
        ('Train', train_clean, '#4FC3F7'),
        ('Val', val_clean, '#F7931A'),
        ('Test', test_clean, '#66BB6A')]):
    y_sample = (part.select(TARGET_COL)
                .sample(withReplacement=False, fraction=0.1, seed=seed)
                .limit(50000).toPandas()[TARGET_COL])
    fig.add_trace(go.Histogram(x=y_sample, name=f'{name} sample', histnorm='probability density',
                               opacity=0.6, marker_color=col))
fig.update_layout(barmode='overlay', title=f'Target Distribution — {TARGET_COL}',
                  template='plotly_dark', xaxis_title='Log Return')
fig.write_html(str(FIGURES / 'target_distribution.html'), include_plotlyjs='cdn')


In [10]:
# ── Save partitions + metadata ────────────────────────────────────────
train_clean.write.mode('overwrite').parquet(str(PROCESSED / 'train.parquet'))
val_clean.write.mode('overwrite').parquet(str(PROCESSED / 'val.parquet'))
test_clean.write.mode('overwrite').parquet(str(PROCESSED / 'test.parquet'))

def max_date(part):
    return part.agg(F.max(TS_COL).alias('end')).first()['end'].date()

split_meta = {
    'feature_cols' : feature_cols,
    'target_col'   : TARGET_COL,
    'ts_col'       : TS_COL,
    'train_end'    : str(max_date(train_clean)),
    'val_end'      : str(max_date(val_clean)),
    'test_end'     : str(max_date(test_clean)),
    'target_horizon': TARGET_HORIZON,
    'bar_interval_seconds': interval_seconds,
    'annualization_factor': ANN_FACTOR,
    'purged_train_rows': purged_train_rows,
    'purged_val_rows': purged_val_rows,
}
with open(PROCESSED / 'split_meta.yaml', 'w') as f:
    yaml.dump(split_meta, f, default_flow_style=False)

print('Partitions saved.')
print('Next → 03_modelling.ipynb')
spark.stop()
print('Spark stopped.')

Partitions saved.
Next → 03_modelling.ipynb
Spark stopped.
